In [1]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
from matplotlib import pyplot as plt
from scipy.stats import pearsonr
from itertools import combinations
import datetime
from scipy.stats import entropy
from collections import Counter
import datetime
import gc
import json

# --- Agent クラス および 補助関数 (変更なし) ---
class Agent:
    def __init__(self, bitN_meaning, bitN_form, m2s, s2m, i):
        self.bitN_meaning = bitN_meaning
        self.bitN_form = bitN_form
        self.m2s = m2s
        self.s2m = s2m
        self.m2m = nn.Sequential(m2s, s2m)
        self.num = i

        
def create_agent(bitN_meaning, bitN_form, nodeN, i):
    m2s = nn.Sequential(
        nn.Linear(bitN_meaning, nodeN), nn.Sigmoid(),
        nn.Linear(nodeN, bitN_form), nn.Sigmoid()
    )
    s2m = nn.Sequential(
        nn.Linear(bitN_form, nodeN), nn.Sigmoid(),
        nn.Linear(nodeN, bitN_meaning), nn.Sigmoid()
    )
    return Agent(bitN_meaning, bitN_form, m2s, s2m, i)


def int2bin(bitN, value):
    return [int(x) for x in f"{value:0{bitN}b}"]


def generate_structured_meaning_space(N_A, N_P, N_R, N_F=1):
    bitN_meaning = N_A + N_P + N_R + N_F
    all_meanings = []
    meaning_pairs = []
    predicates_tensors = [torch.tensor(int2bin(N_R, i), dtype=torch.float32) for i in range(2 ** N_R)]
    agents_tensors = [torch.tensor(int2bin(N_A, i), dtype=torch.float32) for i in range(2 ** N_A)]
    patients_tensors = [torch.tensor(int2bin(N_P, i), dtype=torch.float32) for i in range(2 ** N_P)]
    foci = [torch.tensor([0.0], dtype=torch.float32), torch.tensor([1.0], dtype=torch.float32)]

    for R in predicates_tensors:
        for A in agents_tensors:
            for P in patients_tensors:
                situation_parts = [A, P, R]
                M_act = torch.cat(situation_parts + [foci[0]])
                M_pass = torch.cat(situation_parts + [foci[1]])
                all_meanings.append(M_act)
                all_meanings.append(M_pass)
                meaning_pairs.append((M_act, M_pass))
    return all_meanings, meaning_pairs

def gen_supervised_data(tutor, all_meanings):
    T = []
    for meaning in all_meanings:
        signal = tutor.m2s(meaning.unsqueeze(0)).detach().round().squeeze(0)
        T.append((meaning.numpy(), signal.numpy()))
    return T

def gen_unsupervised_data(all_meanings, A_size):
    U = []
    for _ in range(A_size):
        meaning = random.choice(all_meanings)
        U.append(meaning.numpy())
    return U


def train_combined(agent, tutor, A_size, B_size, all_meanings, epochs, alpha=5.0):
    optimiser_m2s = torch.optim.SGD(agent.m2s.parameters(), lr=5.0)
    optimiser_s2m = torch.optim.SGD(agent.s2m.parameters(), lr=5.0)
    optimiser_m2m = torch.optim.SGD(list(agent.m2s.parameters()) + list(agent.s2m.parameters()), lr=5.0)
    loss_function = nn.MSELoss(reduction='none')
    T = gen_supervised_data(tutor, all_meanings)
    A = gen_unsupervised_data(all_meanings, A_size)
    N_F = 1
    N_SITUATION = agent.bitN_meaning - N_F

    epoch_losses = []

    for epoch in range(epochs):
        total_loss_epoch = 0.0 
        batch_count = 0

        B1 = [random.choice(T) for _ in range(B_size)]
        B2 = B1.copy()
        random.shuffle(B2)

        for i in range(B_size):
            # M -> S
            optimiser_m2s.zero_grad()
            m2s_meaning, m2s_signal = B1[i]
            m2s_meaning = torch.tensor(m2s_meaning, dtype=torch.float32).unsqueeze(0)
            m2s_signal = torch.tensor(m2s_signal, dtype=torch.float32).unsqueeze(0)
            pred_m2s = agent.m2s(m2s_meaning)
            loss_m2s = loss_function(pred_m2s, m2s_signal).mean()
            loss_m2s.backward()
            optimiser_m2s.step()
            
            total_loss_epoch += loss_m2s.item()

            # S -> M
            optimiser_s2m.zero_grad()
            s2m_meaning, s2m_signal = B2[i]
            s2m_signal = torch.tensor(s2m_signal, dtype=torch.float32).unsqueeze(0)
            s2m_meaning = torch.tensor(s2m_meaning, dtype=torch.float32).unsqueeze(0)
            pred_s2m = agent.s2m(s2m_signal)
            loss_s2m = loss_function(pred_s2m, s2m_meaning).mean()
            loss_s2m.backward()
            optimiser_s2m.step()

            total_loss_epoch += loss_s2m.item()

            # Autoencoder (M -> S -> M)
            meanings_u = [random.choice(A) for _ in range(20)]
            for meaning in meanings_u:
                optimiser_m2m.zero_grad()
                auto_m = torch.tensor(meaning, dtype=torch.float32).unsqueeze(0)
                pred_m2m = agent.m2m(auto_m)
                
                loss_elements = loss_function(pred_m2m, auto_m)
                loss_situation = loss_elements[:, :N_SITUATION]
                loss_focus = loss_elements[:, N_SITUATION:]
                weighted_loss_focus = alpha * loss_focus
                loss_auto = torch.cat((loss_situation, weighted_loss_focus), dim=1).mean()
                
                loss_auto.backward()
                optimiser_m2m.step()

                total_loss_epoch += loss_auto.item()
                
            batch_count += 1
        
        epoch_losses.append(total_loss_epoch / batch_count)
                
    return T, epoch_losses

# --- 学習関数2: 参照ゲーム (Referential Game) ---
def train_combined_both(agent, tutor, A_size, B_size, all_meanings, epochs, alpha=5.0, n_distractors=7, comm_weight=1.0):
    """オートエンコーダ(alpha付き, 10回) + 弁別ゲーム(10回) の両方を含む訓練"""
    optimiser_m2s = torch.optim.SGD(agent.m2s.parameters(), lr=5.0)
    optimiser_s2m = torch.optim.SGD(agent.s2m.parameters(), lr=5.0)
    optimiser_m2m = torch.optim.SGD(list(agent.m2s.parameters()) + list(agent.s2m.parameters()), lr=5.0)
    optimiser_comm = torch.optim.SGD(list(agent.m2s.parameters()) + list(agent.s2m.parameters()), lr=1.0)
    loss_function = nn.MSELoss(reduction='none')
    ce_loss = nn.CrossEntropyLoss()
    T = gen_supervised_data(tutor, all_meanings)
    A = gen_unsupervised_data(all_meanings, A_size)
    N_F = 1
    N_SITUATION = agent.bitN_meaning - N_F
    epoch_losses = []

    for epoch in range(epochs):
        total_loss_epoch = 0.0
        batch_count = 0
        B1 = [random.choice(T) for _ in range(B_size)]
        B2 = B1.copy()
        random.shuffle(B2)
        for i in range(B_size):
            # M -> S 教師あり
            optimiser_m2s.zero_grad()
            m, s = B1[i]
            loss_m2s = loss_function(agent.m2s(torch.tensor(m, dtype=torch.float32).unsqueeze(0)),
                                     torch.tensor(s, dtype=torch.float32).unsqueeze(0)).mean()
            loss_m2s.backward(); optimiser_m2s.step()
            total_loss_epoch += loss_m2s.item()

            # S -> M 教師あり
            optimiser_s2m.zero_grad()
            m2, s2 = B2[i]
            loss_s2m = loss_function(agent.s2m(torch.tensor(s2, dtype=torch.float32).unsqueeze(0)),
                                     torch.tensor(m2, dtype=torch.float32).unsqueeze(0)).mean()
            loss_s2m.backward(); optimiser_s2m.step()
            total_loss_epoch += loss_s2m.item()

            # --- オートエンコーダ (alpha付き, 10回) ---
            meanings_u = [random.choice(A) for _ in range(10)]
            for meaning in meanings_u:
                optimiser_m2m.zero_grad()
                auto_m = torch.tensor(meaning, dtype=torch.float32).unsqueeze(0)
                pred_m2m = agent.m2m(auto_m)
                loss_elements = loss_function(pred_m2m, auto_m)
                loss_situation = loss_elements[:, :N_SITUATION]
                loss_focus = loss_elements[:, N_SITUATION:]
                weighted_loss_focus = alpha * loss_focus
                loss_auto = torch.cat((loss_situation, weighted_loss_focus), dim=1).mean()
                loss_auto.backward()
                optimiser_m2m.step()
                total_loss_epoch += loss_auto.item()

            # --- 弁別ゲーム (10回) ---
            for _ in range(10):
                optimiser_comm.zero_grad()
                target = random.choice(all_meanings)
                pool = [m for m in all_meanings if not torch.equal(m, target)]
                distractors = random.sample(pool, n_distractors)
                candidates = [target] + distractors

                signal = agent.m2s(target.unsqueeze(0))
                recon = agent.s2m(signal)

                scores = []
                for c in candidates:
                    dist = torch.sum((recon - c.unsqueeze(0))**2, dim=1)
                    scores.append(-dist)

                scores_tensor = torch.cat(scores).unsqueeze(0)
                loss_comm = comm_weight * ce_loss(scores_tensor, torch.tensor([0]))
                loss_comm.backward(); optimiser_comm.step()
                total_loss_epoch += loss_comm.item()
            batch_count += 1
        epoch_losses.append(total_loss_epoch / batch_count)
    return T, epoch_losses



# --- 学習関数2: 参照ゲーム (Referential Game) ---
def train_combined_with_game(agent, tutor, A_size, B_size, all_meanings, epochs, alpha=5.0, n_distractors=7, comm_weight=1.0):
    optimiser_m2s = torch.optim.SGD(agent.m2s.parameters(), lr=5.0)
    optimiser_s2m = torch.optim.SGD(agent.s2m.parameters(), lr=5.0)
    optimiser_comm = torch.optim.SGD(list(agent.m2s.parameters()) + list(agent.s2m.parameters()), lr=1.0)
    loss_function = nn.MSELoss(reduction='none')
    ce_loss = nn.CrossEntropyLoss()
    T = gen_supervised_data(tutor, all_meanings)
    epoch_losses = []

    for epoch in range(epochs):
        total_loss_epoch = 0.0 
        batch_count = 0
        B1 = [random.choice(T) for _ in range(B_size)]
        B2 = B1.copy()
        random.shuffle(B2)
        for i in range(B_size):
            # M -> S 教師あり
            optimiser_m2s.zero_grad()
            m, s = B1[i]
            loss_m2s = loss_function(agent.m2s(torch.tensor(m, dtype=torch.float32).unsqueeze(0)), 
                                     torch.tensor(s, dtype=torch.float32).unsqueeze(0)).mean()
            loss_m2s.backward(); optimiser_m2s.step()
            total_loss_epoch += loss_m2s.item()

            # S -> M 教師あり
            optimiser_s2m.zero_grad()
            m2, s2 = B2[i]
            loss_s2m = loss_function(agent.s2m(torch.tensor(s2, dtype=torch.float32).unsqueeze(0)), 
                                     torch.tensor(m2, dtype=torch.float32).unsqueeze(0)).mean()
            loss_s2m.backward(); optimiser_s2m.step()
            total_loss_epoch += loss_s2m.item()

            # --- Referential Game ---
            for _ in range(20):
                optimiser_comm.zero_grad()
                target = random.choice(all_meanings)
                pool = [m for m in all_meanings if not torch.equal(m, target)]
                distractors = random.sample(pool, n_distractors)
                candidates = [target] + distractors
                
                signal = agent.m2s(target.unsqueeze(0))
                recon = agent.s2m(signal)
                
                scores = []
                for c in candidates:
                    dist = torch.sum((recon - c.unsqueeze(0))**2, dim=1)
                    scores.append(-dist)
                
                scores_tensor = torch.cat(scores).unsqueeze(0)
                loss_comm = comm_weight * ce_loss(scores_tensor, torch.tensor([0]))
                loss_comm.backward(); optimiser_comm.step()
                total_loss_epoch += loss_comm.item()
            batch_count += 1
        epoch_losses.append(total_loss_epoch / batch_count)
    return T, epoch_losses



# --- ヒートマップ描画関数 (変更なし) ---
def plot_heatmap(agent, all_meanings, save_path, gen, alpha):
    """
    意味ビット(行)と信号ビット(列)の間の相互情報量(MI)をヒートマップで表示・保存する。
    """
    agent.m2s.eval()
    meaning_vectors_list = []
    form_vectors_list = []
    
    with torch.no_grad():
        for meaning_tensor in all_meanings:
            meaning_vectors_list.append(meaning_tensor.numpy())
            signal_tensor = agent.m2s(meaning_tensor.unsqueeze(0)).round().squeeze(0)
            form_vectors_list.append(signal_tensor.numpy())
            
    meanings = np.stack(meaning_vectors_list, axis=0) # (DataSize, M_bits)
    signals = np.stack(form_vectors_list, axis=0)     # (DataSize, S_bits)
    
    n_m = meanings.shape[1]
    n_s = signals.shape[1]
    
    mi_matrix = np.zeros((n_m, n_s))
    
    # 全ビットペア間の相互情報量を計算
    for i in range(n_m):
        for j in range(n_s):
            mi_matrix[i, j] = I(meanings[:, i], signals[:, j]) 
            
    # プロット
    plt.figure(figsize=(8, 6))
    plt.imshow(mi_matrix, cmap='viridis', aspect='auto', vmin=0, vmax=1.0)
    plt.colorbar(label='Mutual Information (bits)')
    
    plt.yticks(ticks=np.arange(n_m), labels=[f"M{i}" for i in range(n_m)])
    plt.xticks(ticks=np.arange(n_s), labels=[f"S{j}" for j in range(n_s)])
    plt.ylabel("Meaning Bits (M)")
    plt.xlabel("Signal Bits (S)")
    plt.title(f"Meaning-Signal Correlation (Gen {gen}, alpha={alpha})")
    
    if save_path:
        os.makedirs(save_path, exist_ok=True)
        filename = f"heatmap_gen{gen}_alpha{alpha}.png"
        plt.savefig(os.path.join(save_path, filename), dpi=300)
    
    plt.close()
    
def _get_meaning_signal_arrays(agent, all_meanings):
    agent.m2s.eval()
    m_list, s_list = [], []
    with torch.no_grad():
        for m in all_meanings:
            m_list.append(m.numpy())
            s = agent.m2s(m.unsqueeze(0)).round().squeeze(0).numpy()
            s_list.append(s)
    return np.stack(m_list), np.stack(s_list)

def calc_mi_matrix(M_array, S_array):
    n_m, n_s = M_array.shape[1], S_array.shape[1]
    matrix = np.zeros((n_m, n_s))
    for i in range(n_m):
        for j in range(n_s):
            matrix[i, j] = I(M_array[:, i], S_array[:, j])
    return matrix

def pearsonr_simple(x, y):
    if np.std(x) == 0 or np.std(y) == 0: return 0
    return pearsonr(x, y)[0]

def MI(v1, v2):
    return I(v1, v2)

def calc_pattern_difference(agent, all_meanings):
    """
    Measure how different the meaning→signal mapping patterns are
    between Focus=0 and Focus=1.

    Computes MI(M_i, S_j) matrix for each Focus group,
    then returns 1 - pearson_correlation between the two matrices.

    Returns: (pattern_diff, pattern_corr, mi_matrix_f0, mi_matrix_f1)
        pattern_diff = 0: identical patterns (like m8_f8)
        pattern_diff > 0: different patterns (Focus-conditional alternation)
    """
    M, S = _get_meaning_signal_arrays(agent, all_meanings)
    focus = M[:, -1]
    M_nf = M[:, :-1]

    idx0 = np.where(focus == 0)[0]
    idx1 = np.where(focus == 1)[0]

    mi_f0 = calc_mi_matrix(M_nf[idx0], S[idx0])
    mi_f1 = calc_mi_matrix(M_nf[idx1], S[idx1])

    flat0 = mi_f0.flatten()
    flat1 = mi_f1.flatten()
    corr = pearsonr_simple(flat0, flat1)

    if np.isnan(corr):
        return 0.0, float('nan'), mi_f0, mi_f1

    return 1.0 - corr, corr, mi_f0, mi_f1


def calc_best_signal_agreement(agent, all_meanings):
    """
    For each non-Focus meaning bit, find the signal bit with highest MI,
    separately for Focus=0 and Focus=1.
    Returns: fraction of meaning bits where the best signal bit is the SAME.
    """
    M, S = _get_meaning_signal_arrays(agent, all_meanings)
    focus = M[:, -1]
    M_nf = M[:, :-1]

    idx0 = np.where(focus == 0)[0]
    idx1 = np.where(focus == 1)[0]

    mi_f0 = calc_mi_matrix(M_nf[idx0], S[idx0])
    mi_f1 = calc_mi_matrix(M_nf[idx1], S[idx1])

    n_m = M_nf.shape[1]
    same_count = sum(1 for i in range(n_m) if np.argmax(mi_f0[i]) == np.argmax(mi_f1[i]))
    return same_count / n_m


# --- Information retention per category ---

def calc_info_retention(agent, all_meanings, category_ranges):
    """
    For each semantic category, compute how much information is retained
    in the signal: sum_j MI(Category, S_j) / H(Category)
    """
    M, S = _get_meaning_signal_arrays(agent, all_meanings)
    n_s = S.shape[1]
    retentions = {}
    for cat_name, (start, end) in category_ranges.items():
        cat_bits = M[:, start:end]
        cat_values = tuple(int("".join(str(int(b)) for b in row), 2) for row in cat_bits)
        total_mi = sum(MI(cat_values, tuple(S[:, j].astype(int))) for j in range(n_s))
        H_cat = H(cat_values)
        retentions[cat_name] = total_mi / H_cat if H_cat > 0 else 0.0
    return retentions

def plot_info_retention(all_retention_by_rep, generations, save_path):
    """
    Semantic Categoryごとの情報保持率を1つのグラフに描画する
    """
    gens = np.arange(1, generations + 1)
    categories = all_retention_by_rep[0][0].keys() # Agent, Patient...
    
    plt.figure(figsize=(8, 5))
    colors = ['blue', 'green', 'orange', 'red']
    
    for cat, col in zip(categories, colors):
        # 全レプリケートの該当カテゴリの値を集計
        cat_values = []
        for rep in all_retention_by_rep:
            cat_values.append([step[cat] for step in rep])
        
        mean_vals = np.mean(cat_values, axis=0)
        plt.plot(gens, mean_vals, label=cat, color=col, linewidth=2.5)
        
    plt.xlabel("Generations")
    plt.ylabel("Information Retention")
    plt.ylim(0, 1.2) # MI/H なので1を超える場合もありますが目安として
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    if save_path:
        os.makedirs(save_path, exist_ok=True)
        plt.savefig(os.path.join(save_path, "info_retention_average.png"), dpi=300)
    plt.close()

# --- 条件付きTopSim (Focus=0, Focus=1 それぞれで計算) ---
def calc_conditional_metrics(agent, all_meanings):
    M, S = _get_meaning_signal_arrays(agent, all_meanings)
    focus = M[:, -1]
    M_nf = M[:, :-1]  # Focusビットを除外
    idx0 = np.where(focus == 0)[0]
    idx1 = np.where(focus == 1)[0]
    # Focus=0群のTopSim
    ts_f0 = TopSim_from_arrays(M_nf[idx0], S[idx0])
    # Focus=1群のTopSim
    ts_f1 = TopSim_from_arrays(M_nf[idx1], S[idx1])
    pd_f0 = PosDis_from_arrays(M_nf[idx0], S[idx0])
    pd_f1 = PosDis_from_arrays(M_nf[idx1], S[idx1])
    comp_f0 = compositionality_from_arrays(M_nf[idx0], S[idx0])
    comp_f1 = compositionality_from_arrays(M_nf[idx1], S[idx1])
    return ts_f0, ts_f1, pd_f0, pd_f1, comp_f0, comp_f1

def TopSim_from_arrays(meaning_vectors, form_vectors):
    """numpy配列を直接受け取るTopSim"""
    if len(meaning_vectors) < 2:
        return np.nan
    m_dists = count_meaning_distance_process(meaning_vectors)
    f_dists = count_form_distance_process(form_vectors)
    if len(set(f_dists)) <= 1:
        return np.nan
    r, _ = pearsonr(m_dists, f_dists)
    return r

def PosDis_from_arrays(meaning_vectors, form_vectors):
    """numpy配列を直接受け取るPosDis"""
    if len(meaning_vectors) < 2:
        return np.nan
    all_messages = form_vectors
    all_attributes = meaning_vectors
    N, MESSAGE_LEN = all_messages.shape
    _, ATTRIBUTES_DIM = all_attributes.shape
    scores = []
    for j in range(MESSAGE_LEN):
        s_j = all_messages[:, j]
        H_s_j = H(s_j)
        if H_s_j == 0:
            continue
        I_scores = []
        for i in range(ATTRIBUTES_DIM):
            a_i = all_attributes[:, i]
            I_scores.append((I(s_j, a_i), i))
        if len(I_scores) < 2:
            continue
        I_scores.sort(key=lambda x: x[0], reverse=True)
        scores.append((I_scores[0][0] - I_scores[1][0]) / H_s_j)
    return np.mean(scores) if scores else 0.0

def compositionality_from_arrays(meaning_vectors, form_vectors):
    """numpy配列を直接受け取るCompositionality"""
    if len(meaning_vectors) < 2:
        return np.nan
    n_m = meaning_vectors.shape[1]
    n_f = form_vectors.shape[1]
    num_messages = len(meaning_vectors)
    meaning_matrix = meaning_vectors.T.astype(int)   # (n_m, N)
    signal_matrix = form_vectors.T.astype(int)        # (n_f, N)
    fact_min_entropies = np.zeros(n_m)
    fact_best_word = np.zeros(n_m, dtype=int)
    for i in range(n_m):
        min_entropy = np.inf
        best_j = -1
        for j in range(n_f):
            p = np.sum(meaning_matrix[i] * signal_matrix[j]) / (num_messages / 2)
            h_ij = calculate_entropy(p)
            if h_ij < min_entropy:
                min_entropy = h_ij
                best_j = j
        fact_min_entropies[i] = min_entropy
        fact_best_word[i] = best_j
    adjusted_entropies = fact_min_entropies.copy()
    for j in range(n_f):
        facts_using_j = np.where(fact_best_word == j)[0]
        if len(facts_using_j) > 1:
            best_fact = facts_using_j[np.argmin(fact_min_entropies[facts_using_j])]
            for idx in facts_using_j:
                if idx != best_fact:
                    adjusted_entropies[idx] = 1.0
    return 1 - np.mean(adjusted_entropies)
    
# --- iterated_learning: 10世代ごとにヒートマップ保存 (変更なし) ---
def iterated_learning(generations, N_A, N_P, N_R, N_F, bitN_form, nodeN, A_size, B_size, epochs, alpha, experiment_root, rep_i, mode="original"):
    
    bitN_meaning = N_A + N_P + N_R + N_F
    tutor = create_agent(bitN_meaning, bitN_form, nodeN, 1)

    stability_scores = []
    expressivity_scores = []
    compositionality_scores = []
    alternation_scores = []
    topsim_scores = [] 
    posdis_scores = []
    generation_losses = []
    
    all_meanings, meaning_pairs = generate_structured_meaning_space(N_A, N_P, N_R, N_F)
    all_meaning_signal_pairs = []

    # ヒートマップ保存用フォルダ (rep_i/heatmaps/)
    heatmap_save_dir = os.path.join(experiment_root, f"rep_{rep_i}", "heatmaps")
    
    pattern_diff_scores = []  # 追加
    best_signal_agreement_scores = []  # 追加
    topsim_f0_scores = []  # 追加: 条件付きTopSim
    topsim_f1_scores = []  # 追加
    posdis_f0_scores = []
    posdis_f1_scores = []
    comp_f0_scores = []
    comp_f1_scores = []
    info_retention_history = []  # 追加 (各世代の辞書を保存)
    # カテゴリの定義 (info_retention用)
    category_ranges = {
        "Agent": (0, N_A),
        "Patient": (N_A, N_A + N_P),
        "Relation": (N_A + N_P, N_A + N_P + N_R),
        "Focus": (N_A + N_P + N_R, N_A + N_P + N_R + N_F)}

    for gen in range(1, generations + 1):
        pupil = create_agent(bitN_meaning, bitN_form, nodeN, gen)
        # モードによって学習関数を切り替え
        if mode == "original":
            current_T, epoch_losses = train_combined(pupil, tutor, A_size, B_size, all_meanings, epochs, alpha=alpha)
        else:
            current_T, epoch_losses = train_combined_both(pupil, tutor, A_size, B_size, all_meanings, epochs, alpha=alpha)
        
        # current_T, epoch_losses = train_combined(pupil, tutor, A_size, B_size, all_meanings, epochs, alpha=alpha)
        
        all_meaning_signal_pairs.append(current_T)

        stability_scores.append(stability(tutor, pupil, all_meanings))
        expressivity_scores.append(expressivity(pupil, all_meanings))
        compositionality_scores.append(compositionality(pupil, all_meanings))
        alternation_scores.append(alternation(pupil, all_meanings, meaning_pairs))
        topsim_scores.append(TopSim(pupil, all_meanings)) 
        posdis_scores.append(PosDis(pupil, all_meanings))
        generation_losses.append(epoch_losses[-1])
        # ★追加した指標の計算
        p_diff, _, _, _ = calc_pattern_difference(pupil, all_meanings)
        pattern_diff_scores.append(p_diff)
        
        best_sa = calc_best_signal_agreement(pupil, all_meanings)
        best_signal_agreement_scores.append(best_sa)
        
        ts_f0, ts_f1, pd_f0, pd_f1, comp_f0, comp_f1 = calc_conditional_metrics(pupil, all_meanings)
        topsim_f0_scores.append(ts_f0)
        topsim_f1_scores.append(ts_f1)
        posdis_f0_scores.append(pd_f0)
        posdis_f1_scores.append(pd_f1)
        comp_f0_scores.append(comp_f0)
        comp_f1_scores.append(comp_f1)
        
        retention = calc_info_retention(pupil, all_meanings, category_ranges)
        info_retention_history.append(retention)

        tutor = pupil
        
        # 10世代ごとにヒートマップを保存
        if gen % 10 == 0: 
             plot_heatmap(pupil, all_meanings, heatmap_save_dir, gen, alpha)

    return (np.array(stability_scores), 
            np.array(expressivity_scores), 
            np.array(compositionality_scores), 
            np.array(alternation_scores), 
            np.array(topsim_scores), 
            np.array(posdis_scores), 
            np.array(generation_losses),
            all_meaning_signal_pairs,
            np.array(pattern_diff_scores), # 追加
            np.array(best_signal_agreement_scores),   # 追加
            np.array(topsim_f0_scores),               # 追加
            np.array(topsim_f1_scores),               # 追加
            np.array(posdis_f0_scores),     # 追加
            np.array(posdis_f1_scores),     # 追加
            np.array(comp_f0_scores),       # 追加
            np.array(comp_f1_scores),       # 追加
            info_retention_history)        # 追加

# --- スコア計算関数群 (省略、変更なし) ---
def stability(tutor, pupil, all_meanings):
    tutor.m2s.eval()
    pupil.s2m.eval()
    matches = 0
    total_meanings = len(all_meanings)
    with torch.no_grad():
        for meaning in all_meanings:
            m = meaning.clone().detach().float().unsqueeze(0)
            tutor_m2s_sig = tutor.m2s(m)
            pupil.s2m.eval()
            pupil_s2m_mn = pupil.s2m(tutor_m2s_sig)
            original_arr = meaning.numpy() > 0.5
            decoded_arr = pupil_s2m_mn.squeeze(0).numpy() > 0.5
            if np.array_equal(original_arr, decoded_arr):
                matches += 1
    return matches / total_meanings

def expressivity(agent, all_meanings):
    agent.m2s.eval()
    unique_signals = set()
    with torch.no_grad():
        for meaning in all_meanings:
            signal = tuple(agent.m2s(meaning).round().squeeze(0).numpy().astype(int))
            unique_signals.add(signal)
    return len(unique_signals) / (2 ** agent.bitN_form)

def calculate_entropy(p):
    if p <= 0 or p >= 1:
        return 0.0
    return -p * np.log2(p) - (1 - p) * np.log2(1 - p)

def compositionality(agent, all_meanings):
    agent.m2s.eval()
    n_m = agent.bitN_meaning
    n_f = agent.bitN_form
    num_messages = len(all_meanings)
    meaning_matrix = np.zeros((n_m, num_messages), dtype=int)
    signal_matrix = np.zeros((n_f, num_messages), dtype=int)

    cnt = 0
    with torch.no_grad():
        for m in all_meanings:
            s = agent.m2s(m.unsqueeze(0)).detach().round().squeeze(0)
            meaning_matrix[:, cnt] = m.numpy()
            signal_matrix[:, cnt] = s.numpy()
            cnt += 1

    fact_min_entropies = np.zeros(n_m)
    fact_best_word = np.zeros(n_m, dtype=int)

    for i in range(n_m):
        min_entropy = np.inf
        best_j = -1
        for j in range(n_f):
            p = np.sum(meaning_matrix[i, :] * signal_matrix[j, :]) / (num_messages / 2)
            h_ij = calculate_entropy(p)

            if h_ij < min_entropy:
                min_entropy = h_ij
                best_j = j
        fact_min_entropies[i] = min_entropy
        fact_best_word[i] = best_j

    adjusted_entropies = fact_min_entropies.copy()
    for j in range(n_f):
        facts_using_j = np.where(fact_best_word == j)[0]
        if len(facts_using_j) > 1:
            best_fact = facts_using_j[np.argmin(fact_min_entropies[facts_using_j])]

            for idx in facts_using_j:
                if idx != best_fact:
                    adjusted_entropies[idx] = 1.0

    average_adjusted_entropy = np.mean(adjusted_entropies)
    return 1 - average_adjusted_entropy

def alternation(agent, all_meanings, meaning_pairs):
    agent.m2s.eval()
    matches = 0
    total_pairs = len(meaning_pairs)
    
    with torch.no_grad():
        for M_act, M_pass in meaning_pairs:
            S_act = agent.m2s(M_act.unsqueeze(0)).round().squeeze(0).numpy().astype(int)
            S_pass = agent.m2s(M_pass.unsqueeze(0)).round().squeeze(0).numpy().astype(int)
            
            if not np.array_equal(S_act, S_pass):
                matches += 1
                
    return matches / total_pairs

def count_meaning_distance_ability(v1, v2):
    distance = np.sum(v1 != v2)
    return distance

def count_meaning_distance_process(meaning_vectors):
    meaning_distance_list = []
    pairs = combinations(meaning_vectors, 2)
    for v1, v2 in pairs:
        distance = count_meaning_distance_ability(v1, v2)
        meaning_distance_list.append(distance)
    return meaning_distance_list

def count_form_distance_ability(v1, v2):
    count = np.sum(v1 != v2)
    return count

def count_form_distance_process(form_vectors):
    form_distance_list = []
    pairs = combinations(form_vectors, 2)
    for v1, v2 in pairs:
        distance = count_form_distance_ability(v1, v2)
        form_distance_list.append(distance)
    return form_distance_list

# TopSim
def TopSim(agent, all_meanings):
    agent.m2s.eval()
    meaning_vectors = []
    form_vectors = []
    
    with torch.no_grad(): 
        for meaning_tensor in all_meanings:
            meaning_array = meaning_tensor.numpy()
            meaning_vectors.append(meaning_array)
            
            signal_tensor = agent.m2s(meaning_tensor.unsqueeze(0)).round().squeeze(0)
            form_array = signal_tensor.numpy()
            form_vectors.append(form_array)
    
    if len(meaning_vectors) < 2:
        return np.nan
        
    meaning_distance_list = count_meaning_distance_process(meaning_vectors)
    form_distance_list = count_form_distance_process(form_vectors)
    
    if len(meaning_vectors) < 2:
        return np.nan

    TopSim_value, TopSim_p_value = pearsonr(meaning_distance_list, form_distance_list)
    return TopSim_value

# PosDis 関連
def calculate_probabilities(data_vector):
    counts = Counter(data_vector)
    total = len(data_vector)
    return {val: count / total for val, count in counts.items()}

def calculate_joint_probabilities(var1, var2): 
    joint_counts = Counter(zip(var1, var2))
    total = len(var1)
    return {pair: count / total for pair, count in joint_counts.items()}

def H(data_vector): 
    probabilities = list(calculate_probabilities(data_vector).values())
    return entropy(probabilities, base=2)

def I(var1, var2): 
    joint_probabilities = list(calculate_joint_probabilities(var1, var2).values())
    H_joint = entropy(joint_probabilities, base=2)
    return H(var1) + H(var2) - H_joint

def PosDis(agent, all_meanings):
    agent.m2s.eval()
    meaning_vectors_list = [] 
    form_vectors_list = []    
    
    with torch.no_grad():
        for meaning_tensor in all_meanings:
            meaning_array = meaning_tensor.numpy()
            meaning_vectors_list.append(meaning_array)
            
            signal_tensor = agent.m2s(meaning_tensor.unsqueeze(0)).round().squeeze(0)
            form_array = signal_tensor.numpy()
            form_vectors_list.append(form_array)
    
    all_messages = np.stack(form_vectors_list, axis=0)      
    all_attributes = np.stack(meaning_vectors_list, axis=0) 

    N, MESSAGE_LEN = all_messages.shape
    _, ATTRIBUTES_DIM = all_attributes.shape
    
    posdis_scores = []
    
    for j in range(MESSAGE_LEN): 
        s_j = all_messages[:, j] 
        H_s_j = H(s_j)
        if H_s_j == 0:
            continue
            
        I_scores = []
        for i in range(ATTRIBUTES_DIM): 
            a_i = all_attributes[:, i] 
            
            I_s_j_a_i = I(s_j, a_i) 
            I_scores.append((I_s_j_a_i, i))
            
        if not I_scores or len(I_scores) < 2:
            continue
            
        I_scores.sort(key=lambda x: x[0], reverse=True)
        
        I_aj1 = I_scores[0][0]
        I_aj2 = I_scores[1][0]
        information_gap = I_aj1 - I_aj2
        
        posdis_term = information_gap / H_s_j
        posdis_scores.append(posdis_term)

    if not posdis_scores:
        return 0.0
        
    PosDis_value = np.mean(posdis_scores)
    return PosDis_value


# --- プロット関数 (rep_i/figures/ に格納) ---
def plot_individual_results(score_array, score_name, generations, save_path):
    gens = np.arange(1, generations + 1)
    
    if save_path is not None:
        os.makedirs(save_path, exist_ok=True)
        
    color = {'stability': 'purple', 'expressivity': 'blue', 'compositionality': 'orange', 'alternation': 'red', 'topsim': 'green', 'posdis': 'grey'}.get(score_name, 'black')

    fig = plt.figure(figsize=(6, 4))
    plt.plot(gens, score_array, color=color, linewidth=3)
    
    plt.xlabel("Generations", fontsize=13)
    plt.ylabel(score_name, fontsize=14)
    plt.ylim(0.00, 1.00) 

    if save_path is not None:
        file_path = os.path.join(save_path, f"{score_name}.png")
        plt.savefig(file_path, dpi=300)
    
    plt.close(fig)

# --- 平均プロット関数 (average_figures/ に格納) ---
def plot_average_results(all_scores_by_rep, score_name, generations, save_path):
    gens = np.arange(1, generations + 1)
    
    if save_path is not None:
        os.makedirs(save_path, exist_ok=True)
    
    color = {'stability': 'purple', 'expressivity': 'blue', 'compositionality': 'orange', 'alternation': 'red', 'topsim': 'green', 'posdis': 'grey'}.get(score_name, 'black')

    stacked_scores = np.stack(all_scores_by_rep, axis=0)
    mean_scores = np.mean(stacked_scores, axis=0)
    
    fig = plt.figure(figsize=(6, 4))
    
    # 1. 個別線（薄い線）
    for i in range(stacked_scores.shape[0]):
        plt.plot(gens, stacked_scores[i], color=color, alpha=0.2, linewidth=1.5)
        
    # 2. 平均線（太い線）
    plt.plot(gens, mean_scores, color=color, linewidth=4, label='Average')
    
    plt.xlabel("Generations", fontsize=13)
    plt.ylabel(f"Average {score_name}", fontsize=14)
    plt.ylim(0.00, 1.00) 

    if save_path is not None:
        file_path = os.path.join(save_path, f"{score_name}_average.png")
        plt.savefig(file_path, dpi=300)
    
    plt.close(fig)
    # plt.show()

# --- TXT保存関数 (rep_i/generations/ に保存) ---
def save_data_txt(all_meaning_signal_pairs_by_rep, experiment_root):
    
    for rep_i, gen_data in enumerate(all_meaning_signal_pairs_by_rep):
        
        data_save_dir = os.path.join(experiment_root, f"rep_{rep_i}", "generations")
        os.makedirs(data_save_dir, exist_ok=True)
        
        for gen_i, T in enumerate(gen_data):
            filename = f"data_rep_{rep_i}_gen_{gen_i+1}.txt"
            filepath = os.path.join(data_save_dir, filename)
            
            with open(filepath, 'w') as f:
                if T:
                    len_M = len(T[0][0])
                    len_S = len(T[0][1])
                    header = f"# Meaning bits: {len_M}, Signal bits: {len_S}\n"
                    header += "# Format: [M_0 M_1 ... M_{N_M-1}] -> [S_0 S_1 ... S_{N_S-1}]\n"
                    f.write(header)
                
                for meaning_array, signal_array in T:
                    meaning_str = ' '.join(map(str, meaning_array.astype(int)))
                    signal_str = ' '.join(map(str, signal_array.astype(int)))
                    f.write(f"{meaning_str} -> {signal_str}\n")
    print(f"Generation data saved to individual 'rep_i/generations' folders.")

# --- main関数: 設定変更適用版 ---
def main():
    # ★ 変更: 世代数 100
    generations = 50
    # ★ 変更: 試行回数 10に修正（元の100から修正）
    replicates = 25 
    
    # 実験パラメータ
    N_A, N_P, N_R, N_F, bitN_form = 3, 2, 2, 1, 8
    
    # ★ 変更: alpha を 1.0〜5.0 の5段階に
    alpha_list = [1.0, 2.0, 3.0, 4.0, 5.0] 
    
    # 比較するモードのリスト
    modes = ["original", "both"]
    
    # 評価指標の名前リスト（ここにあらかじめ追加しておきます）
    score_names = ['stability', 'expressivity', 'compositionality', 'alternation', 'topsim', 'posdis', 'loss', 'pattern_difference', 'best_signal_agreement', 'topsim_f0', 'topsim_f1', 'posdis_f0', 'posdis_f1', 'comp_f0', 'comp_f1']

    for mode in modes:
        for alpha in alpha_list:
            print(f"\n================ STARTING EXPERIMENT alpha = {alpha} ================")

            all_stability_scores = []
            all_expressivity_scores = []
            all_compositionality_scores = []
            all_alternation_scores = []
            all_topsim_scores = []
            all_posdis_scores = []
            all_loss_scores = [] 
            all_meaning_signal_pairs_by_rep = [] 
            all_pattern_diff_scores = [] # 追加
            all_best_sa_scores = []      # 追加
            all_topsim_f0_scores = []    # 追加
            all_topsim_f1_scores = []    # 追加
            all_posdis_f0_scores = []
            all_posdis_f1_scores = []
            all_comp_f0_scores = []
            all_comp_f1_scores = []
            all_info_retention = []      # 追加
        
            # 保存パス作成
            now = datetime.datetime.now()
            timestamp = now.strftime("%Y%m%d_%H%M%S")
            bitN_meaning = N_A + N_P + N_R + N_F
            setting_name = f"{mode}_gen{generations}_m{bitN_meaning}_f{bitN_form}_alpha{int(alpha)}"
            experiment_dir_name = f"exp{timestamp}_{setting_name}"
        
            # パス設定 (カレントディレクトリの親フォルダのout配下)
            current_cwd = os.getcwd() 
            evolang2026_dir = os.path.dirname(current_cwd)
            experiment_root = os.path.join(evolang2026_dir, "out", experiment_dir_name)

            print(f"Experiment Root: {experiment_root}")

            for i in range(replicates):
                print(f"  --- Replicate: {i} (alpha={alpha}) ---")

                # (1) 11個の戻り値を受け取る (修正済み)
                # iterated_learningに experiment_root と rep_i を渡すように変更
                stability, expressivity, compositionality, alternation, topsim, posdis, losses, all_meaning_signal_pairs, pattern_diff, best_sa, topsim_f0, topsim_f1, posdis_f0, posdis_f1, comp_f0, comp_f1, retention = iterated_learning(
                    generations=generations, 
                    N_A=N_A, N_P=N_P, N_R=N_R, N_F=N_F, 
                    bitN_form=bitN_form, nodeN=8, A_size=75, B_size=75, epochs=20, # パラメータ明示 
                    alpha=alpha,
                    experiment_root=experiment_root, # ★ 追加引数
                    rep_i=i,                          # ★ 追加引数
                    mode=mode
                )

                all_stability_scores.append(stability)
                all_expressivity_scores.append(expressivity)
                all_compositionality_scores.append(compositionality)
                all_alternation_scores.append(alternation)
                all_topsim_scores.append(topsim)
                all_posdis_scores.append(posdis)
                all_loss_scores.append(losses) 
                all_meaning_signal_pairs_by_rep.append(all_meaning_signal_pairs)
                all_pattern_diff_scores.append(pattern_diff)
                all_best_sa_scores.append(best_sa)
                all_topsim_f0_scores.append(topsim_f0)
                all_topsim_f1_scores.append(topsim_f1)
                all_posdis_f0_scores.append(posdis_f0)
                all_posdis_f1_scores.append(posdis_f1)
                all_comp_f0_scores.append(comp_f0)
                all_comp_f1_scores.append(comp_f1)
                all_info_retention.append(retention)

                # 個別プロット
                rep_folder = os.path.join(experiment_root, f"rep_{i}")
                individual_figures_path = os.path.join(rep_folder, "figures")
                
                # 指標のデータと名前をセットにする（pattern_diffも含む）
                scores = [stability, expressivity, compositionality, alternation, topsim, posdis, losses, pattern_diff, best_sa, topsim_f0, topsim_f1, posdis_f0, posdis_f1, comp_f0, comp_f1]
                
                for score_array, name in zip(scores, score_names):
                    plot_individual_results(score_array, name, generations, individual_figures_path)

            # 平均プロット
            print(f"  --- Saving Average Results (alpha={alpha}) ---")
            average_figures_path = os.path.join(experiment_root, "average_figures")
            all_scores = [all_stability_scores, all_expressivity_scores, all_compositionality_scores, 
                          all_alternation_scores, all_topsim_scores, all_posdis_scores, all_loss_scores, all_pattern_diff_scores, 
                          all_best_sa_scores, all_topsim_f0_scores, all_topsim_f1_scores, all_posdis_f0_scores, all_posdis_f1_scores, all_comp_f0_scores, all_comp_f1_scores]

        
            for scores, name in zip(all_scores, score_names):
                plot_average_results(scores, name, generations, average_figures_path)
            # Info Retention（複数の線）のプロットを呼び出す
            plot_info_retention(all_info_retention, generations, average_figures_path)

            save_data_txt(all_meaning_signal_pairs_by_rep, experiment_root)

if __name__ == "__main__":
    main()


================ STARTING EXPERIMENT alpha = 1.0 ================
Experiment Root: /Users/iwamurairifuki/JAIST_2025/学会・研究会/evolang2026_cowork/out/exp20260326_205249_original_gen50_m8_f8_alpha1
  --- Replicate: 0 (alpha=1.0) ---


/Users/iwamurairifuki/anaconda3/lib/python3.11/site-packages/scipy/stats/_stats_py.py:4781: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))


  --- Replicate: 1 (alpha=1.0) ---
  --- Replicate: 2 (alpha=1.0) ---
  --- Replicate: 3 (alpha=1.0) ---
  --- Replicate: 4 (alpha=1.0) ---
  --- Replicate: 5 (alpha=1.0) ---
  --- Replicate: 6 (alpha=1.0) ---
  --- Replicate: 7 (alpha=1.0) ---
  --- Replicate: 8 (alpha=1.0) ---
  --- Replicate: 9 (alpha=1.0) ---
  --- Replicate: 10 (alpha=1.0) ---
  --- Replicate: 11 (alpha=1.0) ---
  --- Replicate: 12 (alpha=1.0) ---
  --- Replicate: 13 (alpha=1.0) ---
  --- Replicate: 14 (alpha=1.0) ---
  --- Replicate: 15 (alpha=1.0) ---
  --- Replicate: 16 (alpha=1.0) ---
  --- Replicate: 17 (alpha=1.0) ---
  --- Replicate: 18 (alpha=1.0) ---
  --- Replicate: 19 (alpha=1.0) ---
  --- Replicate: 20 (alpha=1.0) ---
  --- Replicate: 21 (alpha=1.0) ---
  --- Replicate: 22 (alpha=1.0) ---
  --- Replicate: 23 (alpha=1.0) ---
  --- Replicate: 24 (alpha=1.0) ---
  --- Saving Average Results (alpha=1.0) ---
Generation data saved to individual 'rep_i/generations' folders.

================ STARTING EXPERIME